# 01 · Data validation
Ingest raw tables, validate schema/keys/timing/financials, and define Step 01 cleaning rules.
**Output:** `outputs/tables/data_quality_summary.csv`, `schema_summary.csv`, clean core tables in `data/processed/clean/`.

## Load data

In [1]:
import sys, hashlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.common import RAW, PROC, TABLES, OUTCOME_DAYS, OFFERS, HIST_CAMPAIGNS, TABLE_NAMES, load_all_raw
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
raw_hash = {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(RAW.glob("*.csv"))}
t = load_all_raw()
EXPECTED = {"campaigns": (8, 6), "customers": (15000, 9), "historical_assignments": (26600, 7), "web_sessions": (18816, 8),
            "email_events": (18567, 6), "orders": (6984, 14), "order_items": (16306, 6), "products": (180, 6),
            "returns": (693, 5), "offer_catalog": (4, 3), "policy_scoring_population": (2776, 3)}
schema = pd.DataFrame([{"table": n, "rows": len(df), "cols": df.shape[1], "expected": EXPECTED[n],
                        "shape_ok": df.shape == EXPECTED[n], "columns": ",".join(df.columns),
                        "dtypes": ",".join(map(str, df.dtypes))} for n, df in t.items()])
schema[["table", "rows", "cols", "expected", "shape_ok"]]

,table,rows,cols,expected,shape_ok
0,campaigns,8,6,"(8, 6)",True
1,customers,15000,9,"(15000, 9)",True
2,historical_assignments,26600,7,"(26600, 7)",True
3,web_sessions,18816,8,"(18816, 8)",True
4,email_events,18567,6,"(18567, 6)",True
5,orders,6984,14,"(6984, 14)",True
6,order_items,16306,6,"(16306, 6)",True
7,products,180,6,"(180, 6)",True
8,returns,693,5,"(693, 5)",True
9,offer_catalog,4,3,"(4, 3)",True


## DQ test registry

In [2]:
dq = []
def check(test, table, n_fail, severity="error", note=""):
    dq.append({"test": test, "table": table, "n_fail": int(n_fail),
               "status": "PASS" if n_fail == 0 else ("WARN" if severity == "warn" else "FAIL"), "note": note})
for n in TABLE_NAMES:
    check("shape_matches_expected", n, int(not schema.set_index("table").loc[n, "shape_ok"]))

## A. Primary-key uniqueness

In [3]:
PK = {"customers": "customer_id", "campaigns": "campaign_id", "historical_assignments": "assignment_id",
      "web_sessions": "session_id", "email_events": "email_event_id", "orders": "order_id", "returns": "return_id",
      "products": "product_id", "offer_catalog": "offer_type"}
for n, k in PK.items():
    check(f"pk_unique:{k}", n, t[n][k].duplicated().sum() + t[n][k].isna().sum())
check("unique_customer_campaign_assignment", "historical_assignments",
      t["historical_assignments"].duplicated(["customer_id", "campaign_id"]).sum())
check("unique_policy_customer", "policy_scoring_population", t["policy_scoring_population"].customer_id.duplicated().sum())

## C. Duplicates (exact full-row) and order_items cleanup

In [4]:
dups = pd.Series({n: df.duplicated().sum() for n, df in t.items()}, name="exact_dup_rows")
oi_raw = t["order_items"]
oi = oi_raw.drop_duplicates().reset_index(drop=True)
n_oi_dup = len(oi_raw) - len(oi)
check("exact_dup_rows_order_items (removed)", "order_items", n_oi_dup, "warn", f"{n_oi_dup} exact duplicate rows removed")
for n in [x for x in TABLE_NAMES if x != "order_items"]:
    check("no_exact_dup_rows", n, dups[n])
check("pk_unique:(order_id,line_number) after dedup", "order_items", oi.duplicated(["order_id", "line_number"]).sum())
print(dups[dups > 0].to_dict(), "| order_items rows:", len(oi_raw), "->", len(oi))

{'order_items': 160} | order_items rows: 16306 -> 16146


## B. Referential integrity

In [5]:
cust, camp = set(t["customers"].customer_id), set(t["campaigns"].campaign_id)
ha, ee, od, rt, ws, ps = (t[n] for n in ["historical_assignments", "email_events", "orders", "returns", "web_sessions",
                                          "policy_scoring_population"])
check("fk:assignment.customer_id", "historical_assignments", (~ha.customer_id.isin(cust)).sum())
check("fk:assignment.campaign_id in CP01-CP07", "historical_assignments", (~ha.campaign_id.isin(HIST_CAMPAIGNS)).sum())
ee_a = ee.merge(ha, on="assignment_id", how="left", suffixes=("", "_a"))
check("fk:email.assignment_id", "email_events", ee_a.send_ts.isna().sum())
check("fk:email.customer/campaign match assignment", "email_events",
      ((ee_a.customer_id != ee_a.customer_id_a) | (ee_a.campaign_id != ee_a.campaign_id_a)).sum())
check("fk:order.customer_id", "orders", (~od.customer_id.isin(cust)).sum())
check("fk:order.campaign_id (non-null)", "orders", (~od.campaign_id.dropna().isin(camp)).sum())
check("fk:order_item.order_id", "order_items", (~oi.order_id.isin(od.order_id)).sum())
check("fk:order_item.product_id", "order_items", (~oi.product_id.isin(t["products"].product_id)).sum())
check("fk:return.order_id", "returns", (~rt.order_id.isin(od.order_id)).sum())
check("fk:web.customer_id (non-null)", "web_sessions", (~ws.customer_id.dropna().isin(cust)).sum())
check("fk:policy.customer_id", "policy_scoring_population", (~ps.customer_id.isin(cust)).sum())
check("policy.campaign_id is future campaign CP08_POLICY", "policy_scoring_population", (ps.campaign_id != "CP08_POLICY").sum())
ca = od[od.campaign_id.notna()].merge(ha, on=["customer_id", "campaign_id"], how="left")
check("campaign_order maps to customer-campaign assignment", "orders", ca.assignment_id.isna().sum())
check("completed orders have >=1 item", "orders", (~od[od.order_status == "completed"].order_id.isin(oi.order_id)).sum())
print("policy customers with historical assignments:", ps.customer_id.isin(ha.customer_id).sum())

policy customers with historical assignments: 0


## D. Missingness

In [6]:
miss = pd.concat({n: df.isna().sum() for n, df in t.items()}).rename("n_missing").reset_index()
miss.columns = ["table", "column", "n_missing"]
miss["rate"] = miss.n_missing / miss.table.map({n: len(df) for n, df in t.items()})
miss = miss[miss.n_missing > 0]
EXPECTED_MISSING = {("customers", "acquisition_channel"), ("customers", "preferred_category"), ("web_sessions", "customer_id"),
                    ("orders", "campaign_id"), ("orders", "offer_type_redeemed")}
unexpected = miss[~miss.apply(lambda r: (r.table, r.column) in EXPECTED_MISSING, axis=1)]
check("no_unexpected_missing_columns", "all", len(unexpected))
check("orders.campaign_id null only for organic_preperiod", "orders",
      (od.campaign_id.isna() != (od.order_source == "organic_preperiod")).sum())
check("offer_type_redeemed null iff offer_redeemed==0", "orders", (od.offer_type_redeemed.isna() != (od.offer_redeemed == 0)).sum())
check("anonymous web sessions (kept raw, excluded from features)", "web_sessions", ws.customer_id.isna().sum(), "warn")
miss

,table,column,n_missing,rate
9,customers,acquisition_channel,120,0.008000
12,customers,preferred_category,180,0.012000
23,web_sessions,customer_id,220,0.011692
39,orders,campaign_id,3474,0.497423
43,orders,offer_type_redeemed,4662,0.667526


## E. Timing checks

In [7]:
cp = t["campaigns"].set_index("campaign_id")
hac = ha.join(cp[["send_ts", "outcome_window_days"]], on="campaign_id", rsuffix="_cp")
check("assignment.send_ts == campaign.send_ts", "historical_assignments", (hac.send_ts != hac.send_ts_cp).sum())
check("outcome_window_days == 14", "campaigns", (cp.outcome_window_days != OUTCOME_DAYS).sum())
hcs = ha.merge(t["customers"][["customer_id", "signup_date"]], on="customer_id")
check("signup_date <= send_ts", "historical_assignments", (hcs.signup_date > hcs.send_ts).sum())
ca["lag_d"] = (ca.order_ts - ca.send_ts).dt.total_seconds() / 86400
check("campaign orders inside [send, send+14d)", "orders", (~ca.lag_d.between(0, OUTCOME_DAYS, inclusive="left")).sum())
org = od[od.order_source == "organic_preperiod"]
check("organic orders precede first campaign send", "orders", (org.order_ts >= cp.send_ts.min()).sum())
wins = cp.loc[HIST_CAMPAIGNS, ["send_ts"]].sort_values("send_ts")
wins["end"] = wins.send_ts + pd.Timedelta(days=OUTCOME_DAYS)
check("no overlapping historical outcome windows", "campaigns", (wins.send_ts.shift(-1) < wins.end).sum())
ee_a["lag_d"] = (ee_a.event_ts - ee_a.send_ts).dt.total_seconds() / 86400
check("email events at/after own send_ts", "email_events", (ee_a.lag_d < 0).sum())
check("email events only for delivered emails", "email_events", (ee_a.email_delivered != 1).sum())
rto = rt.merge(od, on="order_id")
check("return_ts after order_ts", "returns", (rto.return_ts <= rto.order_ts).sum())
pol_ts = ps.policy_as_of_ts.max()
print("policy_as_of_ts:", pol_ts, "| CP08 send:", cp.loc["CP08_POLICY", "send_ts"])
for n, c in [("web_sessions", "session_ts"), ("email_events", "event_ts"), ("orders", "order_ts"), ("returns", "return_ts")]:
    print(f"{n:14s} {t[n][c].min()} -> {t[n][c].max()} | after policy_as_of_ts: {(t[n][c] > pol_ts).sum()}")
print("email lag (days) max:", round(ee_a.lag_d.max(), 2), "| return lag (days):", (rto.return_ts - rto.order_ts).dt.days.agg(["min", "max"]).to_dict())

policy_as_of_ts: 2025-06-20 10:00:00 | CP08 send: 2025-06-27 10:00:00
web_sessions   2024-07-01 06:58:00 -> 2025-06-19 23:45:00 | after policy_as_of_ts: 0
email_events   2025-01-15 11:00:00 -> 2025-05-25 09:00:00 | after policy_as_of_ts: 0
orders         2024-07-01 08:07:00 -> 2025-06-04 08:56:00 | after policy_as_of_ts: 0
returns        2024-07-06 20:56:00 -> 2025-07-01 08:06:00 | after policy_as_of_ts: 17
email lag (days) max: 3.96 | return lag (days): {'min': 3, 'max': 31}


## F. Financial checks

In [8]:
TOL = 0.05
check("valid order_status", "orders", (~od.order_status.isin(["completed", "cancelled"])).sum())
canc = od[od.order_status == "cancelled"]
fin = ["merchandise_gross_revenue", "discount_amount", "shipping_revenue", "net_revenue", "cogs", "shipping_cost"]
check("cancelled orders carry zero financials", "orders", (canc[fin].abs().sum(axis=1) > 0).sum())
check("net = gross - discount + shipping_rev", "orders",
      ((od.net_revenue - (od.merchandise_gross_revenue - od.discount_amount + od.shipping_revenue)).abs() > TOL).sum())
check("non-negative order financials", "orders", (od[fin] < 0).any(axis=1).sum())
check("discount <= gross", "orders", (od.discount_amount > od.merchandise_gross_revenue + TOL).sum())
check("positive quantity/price/cost", "order_items", ((oi.quantity <= 0) | (oi.unit_list_price <= 0) | (oi.unit_cost < 0)).sum())
check("positive refund_amount", "returns", (rt.refund_amount <= 0).sum())
check("returns only on completed orders", "returns", (rto.order_status != "completed").sum())
check("refund <= order net_revenue", "returns", (rto.refund_amount > rto.net_revenue + TOL).sum())
check("refund <= merchandise_gross_revenue", "returns", (rto.refund_amount > rto.merchandise_gross_revenue + TOL).sum(), "warn",
      "refund may include shipping; within net_revenue")
def item_recon(items):
    g = items.assign(g=items.quantity * items.unit_list_price, c=items.quantity * items.unit_cost).groupby("order_id")[["g", "c"]].sum()
    m = od[od.order_status == "completed"].join(g, on="order_id")
    return ((m.merchandise_gross_revenue - m.g).abs() > TOL).sum(), ((m.cogs - m.c).abs() > TOL).sum()
raw_g, raw_c = item_recon(oi_raw); cln_g, cln_c = item_recon(oi)
check("item gross reconciles to order gross (after dedup)", "order_items", cln_g)
check("item cogs reconciles to order cogs (after dedup)", "order_items", cln_c)
print(f"orders failing item reconciliation: raw={raw_g} gross/{raw_c} cogs -> clean={cln_g}/{cln_c}")
rate = (od.discount_amount / od.merchandise_gross_revenue.where(od.merchandise_gross_revenue > 0))
exp_rate = od.offer_type_redeemed.map({"PCT_10": .10, "PCT_20": .20, "FREE_SHIP": 0}).fillna(0)
check("discount rate matches redeemed offer", "orders", ((rate - exp_rate).abs() > 0.005).sum())
check("FREE_SHIP redemptions have zero shipping revenue", "orders", (od[od.offer_type_redeemed == "FREE_SHIP"].shipping_revenue > 0).sum())

orders failing item reconciliation: raw=160 gross/160 cogs -> clean=0/0


## G. Offer values

In [9]:
cat = set(t["offer_catalog"].offer_type)
check("offer_catalog == 4 official actions", "offer_catalog", len(cat ^ set(OFFERS)))
check("assignment.offer_type in catalog", "historical_assignments", (~ha.offer_type.isin(cat)).sum())
check("offer_type_redeemed in catalog (non-null, != NO_OFFER)", "orders",
      (~od.offer_type_redeemed.dropna().isin(cat - {"NO_OFFER"})).sum())
check("redeemed offer == assigned offer", "orders", (ca.offer_type_redeemed.notna() & (ca.offer_type_redeemed != ca.offer_type)).sum())
pd.crosstab(ca.offer_type, ca.offer_type_redeemed.fillna("none"))

offer_type_redeemed,FREE_SHIP,PCT_10,PCT_20,none
offer_type,,,,
FREE_SHIP,712,0,0,8
NO_OFFER,0,0,0,1149
PCT_10,0,1184,0,22
PCT_20,0,0,426,9


## H. DQ summary

In [10]:
dqs = pd.DataFrame(dq)
dqs.to_csv(TABLES / "data_quality_summary.csv", index=False)
schema.to_csv(TABLES / "schema_summary.csv", index=False)
miss.to_csv(TABLES / "missingness_summary.csv", index=False)
print(dqs.status.value_counts().to_dict(), "| tests:", len(dqs))
dqs[dqs.status != "PASS"]

{'PASS': 76, 'WARN': 3} | tests: 79


,test,table,n_fail,status,note
22,exact_dup_rows_order_items (removed),order_items,160,WARN,160 exact duplicate rows removed
51,"anonymous web sessions (kept raw, excluded fro...",web_sessions,220,WARN,
70,refund <= merchandise_gross_revenue,returns,5,WARN,refund may include shipping; within net_revenue


In [11]:
assert (dqs.status != "FAIL").all(), dqs[dqs.status == "FAIL"]

## Apply Step 01 cleaning rules and save clean core
- `order_items`: exact full-row duplicates removed.
- `orders`: cancelled orders retained for audit; `is_completed` flag marks purchase/financial eligibility.
- `web_sessions`: anonymous rows retained; `is_identified` flag; downstream customer features use identified rows only.
- Organic orders (null `campaign_id`) retained for purchase history. All other tables pass through unchanged.

In [12]:
clean = dict(t)
clean["order_items"] = oi
clean["orders"] = od.assign(is_completed=od.order_status.eq("completed"))
clean["web_sessions"] = ws.assign(customer_id=ws.customer_id.astype("Int64"), is_identified=ws.customer_id.notna())
(PROC / "clean").mkdir(exist_ok=True)
for n, df in clean.items():
    df.to_parquet(PROC / "clean" / f"{n}.parquet", index=False)
clean["order_items"].to_csv(PROC / "clean" / "order_items_clean.csv", index=False)
assert raw_hash == {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(RAW.glob("*.csv"))}
print("saved clean tables:", sorted(p.name for p in (PROC / "clean").glob("*.parquet")), "| raw files unchanged")

saved clean tables: ['campaigns.parquet', 'customers.parquet', 'email_events.parquet', 'historical_assignments.parquet', 'offer_catalog.parquet', 'order_items.parquet', 'orders.parquet', 'policy_scoring_population.parquet', 'products.parquet', 'returns.parquet', 'web_sessions.parquet'] | raw files unchanged


## Summary of assumptions / warnings
- 160 exact duplicate `order_items` rows removed; afterwards items reconcile to order gross/COGS within $0.05.
- 220 anonymous web sessions (null `customer_id`) kept raw, excluded from customer-level features.
- 102 cancelled orders have all-zero financials; flagged `is_completed=False`, never counted as purchases.
- Organic preperiod orders (null `campaign_id`) all precede CP01 and feed purchase history only.
- Returns occur 3–31 days after order, so many fall after a campaign's 14-day window (handled explicitly in Step 03).
- A few refunds exceed merchandise gross (but not net revenue) — retained, flagged as WARN.
- Policy-scoring customers have no historical campaign assignments (relevant for Step 07).